# Lesson 11.1: What Are the Right Evaluation Metrics for a RAG System?

**Companion notebook for Lesson 11.1 — Module 11: Evaluation**

---

| Section | What you will build |
|---|---|
| 1. Setup — Golden Dataset | 7 labeled queries with known relevant doc IDs |
| 2. Three Retrieval Strategies | Good / Noisy / Poor — something to measure against |
| 3. Floor 1 — Recall@K | Did we find the right doc anywhere in top-k? |
| 4. Floor 1 — Precision@K | How much of what we retrieved was actually useful? |
| 5. Floor 1 — MRR | How high up was the first relevant result? |
| 6. Floor 1 — NDCG | Graded relevance with rank discounting |
| 7. Floor 1 Comparison | All four metrics across three retrieval strategies |
| 8. High Recall, Broken Answer | Prove Floor 1 alone is not enough |
| 9. Floor 2 — Faithfulness | Did the LLM make stuff up? (claim-level check) |
| 10. Floor 2 — Context Recall | Did retrieval find all the needed information? |
| 11. Floor 2 — Context Precision | Were the relevant chunks ranked at the top? |
| 12. Floor 2 — Answer Relevance | Did we answer the question that was actually asked? |
| 13. Floor 2 — Answer Correctness | Factual accuracy vs. ground-truth answer |
| 14. Two-Metric Diagnostic | Faithfulness + Context Recall: localise which half of the pipeline failed |
| 15. Cheat Sheet Visualisation | Full metric map with when-to-use guide |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`  
**Optional (Floor 2 real LLM mode):** `anthropic`

> Floor 2 LLM-judge cells run in **mock mode** by default.
> Set `USE_REAL_LLM = True` + `ANTHROPIC_API_KEY` for live Claude-as-judge.


In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib
# !pip install anthropic  # optional — Floor 2 real mode

%matplotlib inline
import os, json, math, re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS']        = '1'

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

USE_REAL_LLM = False

ANTHROPIC_AVAILABLE = False
try:
    import anthropic as _ant
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass

if USE_REAL_LLM and not ANTHROPIC_AVAILABLE:
    print('WARNING: USE_REAL_LLM=True but anthropic not available. Falling back to mock.')
    USE_REAL_LLM = False

print(f'Mode: {"REAL" if USE_REAL_LLM else "MOCK"}')
print('Imports ready.')


In [ ]:
# ── Knowledge base ────────────────────────────────────────────────────────────
DOCS = {
    'refund_policy': (
        'Customers may return any product within 30 days of purchase for a full refund. '
        'Digital goods are non-refundable once downloaded. No restocking fee applies. '
        'To initiate a return, contact returns@company.com with your order number.'
    ),
    'shipping': (
        'Standard US shipping costs $9.99 and takes 5-7 business days. '
        'Expedited 2-day shipping is $24.99. Orders over $150 qualify for free standard shipping. '
        'International rates vary by destination. Orders ship within 1 business day of placement.'
    ),
    'enterprise_sla': (
        'Enterprise customers receive a 99.9% uptime SLA. '
        'Support response time is 4 hours for critical issues and 24 hours for non-critical. '
        'Planned maintenance windows are Sundays 02:00-04:00 UTC. '
        'SLA credits apply if uptime falls below 99.5% in any calendar month.'
    ),
    'free_tier': (
        'The free tier includes core features, 2GB of storage, and community forum support. '
        'Free tier users are limited to 100 API calls per day. '
        'There is no credit card required to sign up for the free tier.'
    ),
    'data_privacy': (
        'We do not sell user data to third parties. '
        'Personal data is retained for 24 months after account closure unless deletion is requested. '
        'We are GDPR and CCPA compliant. Users can export or delete their data at any time.'
    ),
    'billing': (
        'Subscriptions are billed monthly on the anniversary of signup. '
        'You can cancel at any time; access continues until the end of the billing period. '
        'Annual subscribers receive a prorated refund if they cancel within 60 days of renewal. '
        'Billing questions: billing@company.com'
    ),
    'api_limits': (
        'Free tier: 100 API calls per day, 10 requests per minute. '
        'Pro tier: 10,000 API calls per day, 500 requests per minute. '
        'Enterprise: unlimited calls, custom rate limits. '
        'Exceeding limits returns HTTP 429. Unused quota does not roll over.'
    ),
}

# ── Golden dataset ─────────────────────────────────────────────────────────────
# Each entry: query, relevant_doc_ids (ground truth), ground_truth_answer
GOLDEN = [
    {
        'id': 'q1',
        'query': 'What is the refund policy?',
        'relevant_docs': ['refund_policy'],
        'ground_truth': (
            'You can return any product within 30 days of purchase for a full refund. '
            'Digital goods are non-refundable. There is no restocking fee. '
            'Contact returns@company.com to initiate a return.'
        ),
    },
    {
        'id': 'q2',
        'query': 'How long does shipping take?',
        'relevant_docs': ['shipping'],
        'ground_truth': (
            'Standard US shipping takes 5-7 business days. '
            'Expedited 2-day shipping is also available. '
            'Orders ship within 1 business day of placement.'
        ),
    },
    {
        'id': 'q3',
        'query': 'What uptime does the enterprise plan guarantee?',
        'relevant_docs': ['enterprise_sla'],
        'ground_truth': (
            'Enterprise customers receive a 99.9% uptime SLA. '
            'SLA credits apply if uptime falls below 99.5% in any calendar month.'
        ),
    },
    {
        'id': 'q4',
        'query': 'How many API calls can I make on the free tier?',
        'relevant_docs': ['api_limits', 'free_tier'],
        'ground_truth': (
            'Free tier users are limited to 100 API calls per day and 10 requests per minute. '
            'Unused quota does not roll over.'
        ),
    },
    {
        'id': 'q5',
        'query': 'Do you sell user data to third parties?',
        'relevant_docs': ['data_privacy'],
        'ground_truth': 'We do not sell user data to third parties.',
    },
    {
        'id': 'q6',
        'query': 'Can I cancel my subscription?',
        'relevant_docs': ['billing'],
        'ground_truth': (
            'Yes, you can cancel at any time. '
            'Access continues until the end of the billing period. '
            'Annual subscribers get a prorated refund if they cancel within 60 days of renewal.'
        ),
    },
    {
        'id': 'q7',
        'query': 'What storage does the free tier include?',
        'relevant_docs': ['free_tier'],
        'ground_truth': 'The free tier includes 2GB of storage.',
    },
]

DOC_IDS = list(DOCS.keys())
print(f'Knowledge base: {len(DOCS)} documents')
print(f'Golden dataset: {len(GOLDEN)} labeled queries')
print()
for g in GOLDEN:
    print(f'  [{g["id"]}] "{g["query"]}" → {g["relevant_docs"]}')


In [ ]:
# ── Three retrieval strategies to compare ────────────────────────────────────
# Each returns a list of (doc_id, relevance_grade) in rank order.
# relevance_grade: 2=highly relevant, 1=somewhat relevant, 0=irrelevant

# Strategy A: Good retriever — correct doc in position 1 or 2
RETRIEVALS_GOOD = {
    'q1': [('refund_policy',2),('shipping',0),('billing',1),('free_tier',0),('enterprise_sla',0)],
    'q2': [('shipping',2),('refund_policy',0),('billing',0),('free_tier',0),('data_privacy',0)],
    'q3': [('enterprise_sla',2),('billing',1),('free_tier',0),('api_limits',0),('shipping',0)],
    'q4': [('api_limits',2),('free_tier',2),('billing',0),('enterprise_sla',1),('shipping',0)],
    'q5': [('data_privacy',2),('billing',0),('free_tier',0),('shipping',0),('refund_policy',0)],
    'q6': [('billing',2),('refund_policy',1),('free_tier',0),('data_privacy',0),('shipping',0)],
    'q7': [('free_tier',2),('api_limits',1),('billing',0),('shipping',0),('data_privacy',0)],
}

# Strategy B: Noisy retriever — correct doc present but buried in noise
RETRIEVALS_NOISY = {
    'q1': [('shipping',0),('billing',0),('refund_policy',2),('free_tier',0),('data_privacy',0)],
    'q2': [('billing',0),('refund_policy',0),('free_tier',0),('shipping',2),('enterprise_sla',0)],
    'q3': [('free_tier',0),('data_privacy',0),('billing',0),('api_limits',0),('enterprise_sla',2)],
    'q4': [('billing',0),('refund_policy',0),('enterprise_sla',0),('api_limits',2),('free_tier',2)],
    'q5': [('shipping',0),('billing',0),('enterprise_sla',0),('refund_policy',0),('data_privacy',2)],
    'q6': [('shipping',0),('free_tier',0),('data_privacy',0),('billing',2),('refund_policy',1)],
    'q7': [('billing',0),('shipping',0),('data_privacy',0),('enterprise_sla',0),('free_tier',2)],
}

# Strategy C: Poor retriever — often misses the correct doc entirely
RETRIEVALS_POOR = {
    'q1': [('shipping',0),('billing',0),('free_tier',0),('data_privacy',0),('enterprise_sla',0)],
    'q2': [('billing',0),('data_privacy',0),('free_tier',0),('enterprise_sla',0),('refund_policy',0)],
    'q3': [('api_limits',0),('free_tier',0),('data_privacy',0),('shipping',0),('billing',0)],
    'q4': [('refund_policy',0),('billing',0),('data_privacy',0),('enterprise_sla',0),('shipping',0)],
    'q5': [('billing',0),('enterprise_sla',0),('api_limits',0),('free_tier',0),('shipping',0)],
    'q6': [('free_tier',0),('api_limits',0),('enterprise_sla',0),('data_privacy',0),('refund_policy',0)],
    'q7': [('refund_policy',0),('billing',0),('enterprise_sla',0),('api_limits',0),('shipping',0)],
}

STRATEGIES = {
    'Good retriever':  RETRIEVALS_GOOD,
    'Noisy retriever': RETRIEVALS_NOISY,
    'Poor retriever':  RETRIEVALS_POOR,
}

def get_retrieved_ids(retrievals, qid, k=5):
    return [doc_id for doc_id, _ in retrievals[qid][:k]]

def get_retrieved_grades(retrievals, qid, k=5):
    return [grade for _, grade in retrievals[qid][:k]]

print('Three retrieval strategies defined.')
print()
print('Example — q1 "What is the refund policy?" (correct doc = refund_policy):')
for name, retr in STRATEGIES.items():
    ranks = [(i+1, did, g) for i, (did, g) in enumerate(retr['q1'])]
    print(f'  {name}:')
    for rank, did, grade in ranks:
        marker = '<-- CORRECT' if grade > 0 else ''
        print(f'    #{rank} [{did}] grade={grade} {marker}')
    print()


---
## Floor 1: Retrieval Metrics

**The cheapest metrics to compute.** No API calls. No LLM judge. Just math on lists of document IDs.

**The catch:** you need a small set of labeled queries — queries where you know which document IDs are the correct answer. This is your *golden dataset*.

```
Query: "What is the refund policy?"
Ground truth relevant docs: [refund_policy]
Retriever returned:         [shipping, billing, refund_policy, free_tier, enterprise_sla]
```

All four Floor 1 metrics answer different questions about this list.


In [ ]:
# ── Recall@K ─────────────────────────────────────────────────────────────────
# Out of ALL documents that should have been retrieved,
# what fraction did we actually find in the top k?
# Formula: |relevant ∩ retrieved@k| / |relevant|

def recall_at_k(retrieved_ids: list, relevant_ids: list) -> float:
    if not relevant_ids:
        return 1.0
    hits = len(set(retrieved_ids) & set(relevant_ids))
    return hits / len(relevant_ids)


def compute_recall(strategy_retrievals, golden, k=5):
    scores = []
    for g in golden:
        retrieved = get_retrieved_ids(strategy_retrievals, g['id'], k)
        scores.append(recall_at_k(retrieved, g['relevant_docs']))
    return scores


print(f'=== Recall@5 across strategies ===\n')
print(f'{"Query":<50} {"Good":<8} {"Noisy":<8} {"Poor"}')
print('-' * 75)

recall_scores = {}
for name, retr in STRATEGIES.items():
    recall_scores[name] = compute_recall(retr, GOLDEN, k=5)

for i, g in enumerate(GOLDEN):
    scores = [recall_scores[n][i] for n in STRATEGIES]
    print(f'{g["query"][:48]:<50} {scores[0]:<8.2f} {scores[1]:<8.2f} {scores[2]:.2f}')

print('-' * 75)
for name, scores in recall_scores.items():
    print(f'  Mean Recall@5  ({name:<18}): {np.mean(scores):.3f}')

print()
print('Recall@K answers: "Did we find the right doc ANYWHERE in the top-k bag?"')
print('Use it when: the LLM only needs one good chunk to answer well.')
print('Blind spot: a perfect recall@5=1.0 still means the right doc might be at #5,')
print('           buried under four irrelevant chunks.')


In [ ]:
# ── Precision@K ──────────────────────────────────────────────────────────────
# Of the k documents we retrieved, how many were actually relevant?
# Formula: |relevant ∩ retrieved@k| / k

def precision_at_k(retrieved_ids: list, relevant_ids: list) -> float:
    if not retrieved_ids:
        return 0.0
    hits = len(set(retrieved_ids) & set(relevant_ids))
    return hits / len(retrieved_ids)


def compute_precision(strategy_retrievals, golden, k=5):
    scores = []
    for g in golden:
        retrieved = get_retrieved_ids(strategy_retrievals, g['id'], k)
        scores.append(precision_at_k(retrieved, g['relevant_docs']))
    return scores


print(f'=== Precision@5 across strategies ===\n')
print(f'{"Query":<50} {"Good":<8} {"Noisy":<8} {"Poor"}')
print('-' * 75)

prec_scores = {}
for name, retr in STRATEGIES.items():
    prec_scores[name] = compute_precision(retr, GOLDEN, k=5)

for i, g in enumerate(GOLDEN):
    scores = [prec_scores[n][i] for n in STRATEGIES]
    print(f'{g["query"][:48]:<50} {scores[0]:<8.2f} {scores[1]:<8.2f} {scores[2]:.2f}')

print('-' * 75)
for name, scores in prec_scores.items():
    print(f'  Mean Precision@5 ({name:<18}): {np.mean(scores):.3f}')

print()
print('Precision@K answers: "How much of what we retrieved was actually useful?"')
print('Use it when: every irrelevant chunk in your prompt window costs you (distraction + tokens).')
print('Note: precision and recall trade off. Retrieving fewer docs often boosts precision')
print('      but may hurt recall.')


In [ ]:
# ── Mean Reciprocal Rank (MRR) ────────────────────────────────────────────────
# If the first relevant doc is at rank r, score = 1/r
# Average across all queries
# MRR rewards finding the right doc as early as possible

def reciprocal_rank(retrieved_ids: list, relevant_ids: list) -> float:
    relevant_set = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0  # relevant doc not found in top-k


def compute_mrr(strategy_retrievals, golden, k=5):
    scores = []
    for g in golden:
        retrieved = get_retrieved_ids(strategy_retrievals, g['id'], k)
        scores.append(reciprocal_rank(retrieved, g['relevant_docs']))
    return scores


print(f'=== MRR (Mean Reciprocal Rank) across strategies ===\n')
print(f'{"Query":<50} {"Good":<8} {"Noisy":<8} {"Poor"}')
print('-' * 75)

mrr_scores = {}
for name, retr in STRATEGIES.items():
    mrr_scores[name] = compute_mrr(retr, GOLDEN, k=5)

for i, g in enumerate(GOLDEN):
    scores = [mrr_scores[n][i] for n in STRATEGIES]
    print(f'{g["query"][:48]:<50} {scores[0]:<8.3f} {scores[1]:<8.3f} {scores[2]:.3f}')

print('-' * 75)
for name, scores in mrr_scores.items():
    print(f'  MRR ({name:<18}): {np.mean(scores):.3f}')

print()
print('MRR answers: "How high up was the FIRST relevant result?"')
print('Scores: rank-1 = 1.0, rank-2 = 0.5, rank-3 = 0.33, rank-5 = 0.2, not found = 0.0')
print('Use it when: the LLM weights the first chunk heavily, or results are shown to users.')
print('The good retriever scores high (1.0 per query); the noisy retriever pays for ranking.')


In [ ]:
# ── NDCG (Normalised Discounted Cumulative Gain) ──────────────────────────────
# MRR's fancier cousin: handles graded relevance (not just binary).
# A doc with grade=2 contributes more than grade=1.
# Positions near the top contribute more than positions further down.
#
# DCG  = sum(grade_i / log2(rank_i + 1))  for i in top-k
# IDCG = DCG of the ideal ranking (grades sorted descending)
# NDCG = DCG / IDCG   (normalised to [0, 1])

def dcg(grades: list) -> float:
    return sum(g / math.log2(i + 2) for i, g in enumerate(grades))

def ndcg_at_k(retrieved_grades: list) -> float:
    actual_dcg = dcg(retrieved_grades)
    ideal_dcg  = dcg(sorted(retrieved_grades, reverse=True))
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0


def compute_ndcg(strategy_retrievals, golden, k=5):
    scores = []
    for g in golden:
        grades = get_retrieved_grades(strategy_retrievals, g['id'], k)
        scores.append(ndcg_at_k(grades))
    return scores


print(f'=== NDCG@5 across strategies ===\n')
print(f'{"Query":<50} {"Good":<8} {"Noisy":<8} {"Poor"}')
print('-' * 75)

ndcg_scores = {}
for name, retr in STRATEGIES.items():
    ndcg_scores[name] = compute_ndcg(retr, GOLDEN, k=5)

for i, g in enumerate(GOLDEN):
    scores = [ndcg_scores[n][i] for n in STRATEGIES]
    print(f'{g["query"][:48]:<50} {scores[0]:<8.3f} {scores[1]:<8.3f} {scores[2]:.3f}')

print('-' * 75)
for name, scores in ndcg_scores.items():
    print(f'  NDCG@5 ({name:<18}): {np.mean(scores):.3f}')

print()
print('NDCG penalises two things simultaneously:')
print('  1. Relevant docs placed at low ranks (rank-discounting)')
print('  2. Highly-relevant docs treated same as weakly-relevant ones (grade weighting)')
print('Use NDCG when you have multi-level relevance labels. Start with MRR/Recall for early RAG.')


In [ ]:
# ── Floor 1: all four metrics side by side ───────────────────────────────────
strategy_names = list(STRATEGIES.keys())
colors = ['#1565C0', '#F57F17', '#E53935']

metric_data = [
    ('Recall@5',    recall_scores),
    ('Precision@5', prec_scores),
    ('MRR',         mrr_scores),
    ('NDCG@5',      ndcg_scores),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
x = np.arange(len(strategy_names))
w = 0.6

for ax, (metric_name, scores_dict) in zip(axes, metric_data):
    means = [np.mean(scores_dict[n]) for n in strategy_names]
    bars  = ax.bar(x, means, w, color=colors, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' ',  '\n') for n in strategy_names], fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel(metric_name)
    ax.set_title(metric_name, fontweight='bold')
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

fig.suptitle('Floor 1 Retrieval Metrics — Three Retrieval Strategies Compared',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print('Key observation: the noisy retriever has recall@5 = 1.0 (it FINDS the right doc)')
print('but MRR = 0.28 (the right doc is buried at rank 3-5 every time).')
print('NDCG captures both failures: missed docs AND poor ranking.')


---
## 8. The Honest Truth: High Recall Does Not Guarantee a Good Answer

The noisy retriever has Recall@5 = 1.0 — it *finds* the right doc every time.
Yet a user asking that system could still get a terrible answer. Why?

Three ways high recall still fails:

1. **The LLM ignores the right chunk and hallucinates** — it's buried at rank 5
   under four irrelevant chunks that confused the model.

2. **The right chunk is there but the LLM misread it** — LLMs misinterpret numbers
   and dates more than they misinterpret concepts.

3. **The retrieved text is relevant but incomplete** — the question has two parts;
   retrieval found the answer to part one but missed part two.

This is why we need Floor 2. Let's go upstairs.


---
## Floor 2: RAG-Specific Metrics (LLM as Judge)

Floor 2 asks questions that pure math cannot answer:
*"Is this answer actually supported by the context?"*

To compute them, you use another LLM as a judge. This is the trick that lets you
evaluate **without labeling thousands of examples by hand**.

The four core metrics form a 2×2 grid:

```
              About the CONTEXT          About the ANSWER
              ─────────────────          ────────────────
About the     Context Recall             Faithfulness
  OUTPUT      (retrieval found           (answer is grounded
               what was needed)           in the context)

About the     Context Precision          Answer Relevance
  RANKING     (relevant chunks           (answer addresses
               ranked at top)             the right question)
```

We also compute **Answer Correctness** (requires ground-truth labels).

In mock mode below: heuristics approximate what an LLM judge would do.
In real mode (`USE_REAL_LLM=True`): Claude acts as the judge.


In [ ]:
# ── Simulated RAG answers ────────────────────────────────────────────────────
# Three answer quality levels:
# - faithful_answers: grounded in context, correct
# - hallucinated_answers: contain fabricated claims
# - off_topic_answers: real but not addressing the question asked

FAITHFUL_ANSWERS = {
    'q1': ('You can return any product within 30 days of purchase for a full refund. '
           'Digital goods are non-refundable. There is no restocking fee. '
           'Contact returns@company.com to start a return.'),
    'q2': ('Standard US shipping takes 5 to 7 business days and costs $9.99. '
           'Expedited 2-day shipping is also available. '
           'Orders ship within 1 business day of placement.'),
    'q3': ('Enterprise customers receive a 99.9% uptime SLA. '
           'If uptime falls below 99.5%, SLA credits apply.'),
    'q4': ('Free tier users can make 100 API calls per day and 10 requests per minute. '
           'Unused quota does not roll over.'),
    'q5': ('We do not sell user data to third parties.'),
    'q6': ('You can cancel at any time. Access continues until the end of the billing period. '
           'Annual subscribers can get a prorated refund if they cancel within 60 days of renewal.'),
    'q7': ('The free tier includes 2GB of storage.'),
}

HALLUCINATED_ANSWERS = {
    'q1': ('You can return any product within 90 days of purchase for a full refund. '
           'There is a 15% restocking fee on all returns. '
           'Contact support@company.com to initiate a return.'),
    # 90 days (should be 30), 15% fee (there is NO fee), wrong email
    'q2': ('Standard shipping is free for all orders and takes 2-3 business days. '
           'International shipping takes 7-10 days.'),
    # free (not free), 2-3 days (should be 5-7)
    'q3': ('Enterprise customers get a 99.99% uptime SLA with a 1-hour critical response time.'),
    # 99.99% (should be 99.9%), 1-hour (should be 4-hour)
    'q4': ('Free tier users can make 500 API calls per day. Unused quota rolls over monthly.'),
    # 500 (should be 100), rollover is false
    'q5': ('We share anonymised data with trusted advertising partners.'),
    # completely fabricated — we do NOT sell/share data
    'q6': ('Cancellations take effect immediately and you lose access right away.'),
    # wrong — access continues until end of billing period
    'q7': ('The free tier includes 10GB of storage and 1TB of bandwidth.'),
    # 10GB (should be 2GB), bandwidth claim is fabricated
}

OFF_TOPIC_ANSWERS = {
    'q1': ('Our company offers a range of products at competitive prices. '
           'Customer satisfaction is our top priority.'),
    'q2': ('We have multiple shipping partners to ensure reliable delivery.'),
    'q3': ('Enterprise plans offer many advanced features for large organisations.'),
    'q4': ('Our API is designed to be developer-friendly with comprehensive documentation.'),
    'q5': ('We take privacy very seriously and are committed to protecting user data.'),
    'q6': ('Our subscription plans offer flexible billing options to suit your needs.'),
    'q7': ('The free tier is a great way to try our product before committing.'),
}

ANSWER_SETS = {
    'Faithful':    FAITHFUL_ANSWERS,
    'Hallucinated': HALLUCINATED_ANSWERS,
    'Off-topic':   OFF_TOPIC_ANSWERS,
}

print('Answer sets ready:')
for name, answers in ANSWER_SETS.items():
    print(f'  {name}: {len(answers)} answers')
print()
print('Example q1 ("What is the refund policy?"):')
for name, answers in ANSWER_SETS.items():
    print(f'  {name}: "{answers["q1"][:80]}..."')


In [ ]:
from sentence_transformers import SentenceTransformer, util

EMB_MODEL = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print('Embedding model loaded.')


def faithfulness_mock(answer: str, context: str, threshold: float = 0.45) -> dict:
    """
    Mock faithfulness judge.
    Splits answer into sentences (claims).
    For each claim, embeds it and checks max cosine similarity to context sentences.
    If similarity < threshold, the claim is 'not supported'.
    Returns: {score, supported, total, unsupported_claims}
    """
    claims   = [s.strip() for s in re.split(r'(?<=[.!?])\s+', answer) if len(s.strip()) > 10]
    ctx_sents= [s.strip() for s in re.split(r'(?<=[.!?])\s+', context) if len(s.strip()) > 5]

    if not claims or not ctx_sents:
        return {'score': 0.0, 'supported': 0, 'total': len(claims), 'unsupported': []}

    claim_embs = EMB_MODEL.encode(claims,    convert_to_tensor=True, show_progress_bar=False)
    ctx_embs   = EMB_MODEL.encode(ctx_sents, convert_to_tensor=True, show_progress_bar=False)

    supported = 0
    unsupported = []
    for i, claim in enumerate(claims):
        max_sim = float(util.cos_sim(claim_embs[i], ctx_embs).max())
        if max_sim >= threshold:
            supported += 1
        else:
            unsupported.append(claim)

    score = supported / len(claims) if claims else 0.0
    return {'score': score, 'supported': supported,
            'total': len(claims), 'unsupported': unsupported}


def faithfulness_llm(answer: str, context: str) -> dict:
    """Real LLM judge version."""
    client = _ant.Anthropic()
    claims = [s.strip() for s in re.split(r'(?<=[.!?])\s+', answer) if len(s.strip()) > 10]
    supported = 0
    unsupported = []
    for claim in claims:
        prompt = (
            f'Context: {context}\n\n'
            f'Claim: {claim}\n\n'
            'Is this claim fully supported by the context? '
            'Answer with only "yes" or "no".'
        )
        resp = client.messages.create(
            model='claude-haiku-4-5-20251001', max_tokens=5,
            messages=[{'role': 'user', 'content': prompt}])
        verdict = resp.content[0].text.strip().lower()
        if 'yes' in verdict:
            supported += 1
        else:
            unsupported.append(claim)
    score = supported / len(claims) if claims else 0.0
    return {'score': score, 'supported': supported,
            'total': len(claims), 'unsupported': unsupported}


def faithfulness(answer, context):
    return faithfulness_llm(answer, context) if USE_REAL_LLM else faithfulness_mock(answer, context)


print()
print('=== Faithfulness scores ===\n')
faith_scores = defaultdict(dict)
for qid_obj in GOLDEN[:3]:  # demo on first 3 queries
    qid     = qid_obj['id']
    context = DOCS[qid_obj['relevant_docs'][0]]
    print(f'[{qid}] "{qid_obj["query"][:50]}"')
    for ans_name, answers in ANSWER_SETS.items():
        result = faithfulness(answers[qid], context)
        faith_scores[ans_name][qid] = result['score']
        print(f'  {ans_name:<15} score={result["score"]:.2f}  '
              f'({result["supported"]}/{result["total"]} claims supported)')
        if result['unsupported']:
            print(f'    Unsupported: "{result["unsupported"][0][:70]}..."')
    print()

print('Faithfulness = #1 metric to instrument.')
print('Hallucinations score low; faithful answers score high; off-topic answers are also "faithful"')
print('(they do not claim things that are false — they just do not say enough).')


In [ ]:
# ── Context Recall ───────────────────────────────────────────────────────────
# Out of all the information needed to answer the question (from ground truth),
# how much was actually present in the retrieved context?
# Requires ground-truth answers.

def context_recall_mock(ground_truth: str, context: str,
                        threshold: float = 0.45) -> dict:
    """
    Break ground-truth answer into claims.
    For each claim, check if it is semantically covered by any context sentence.
    """
    gt_claims = [s.strip() for s in re.split(r'(?<=[.!?])\s+', ground_truth)
                 if len(s.strip()) > 10]
    ctx_sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', context)
                 if len(s.strip()) > 5]

    if not gt_claims or not ctx_sents:
        return {'score': 0.0, 'covered': 0, 'total': len(gt_claims), 'missing': []}

    gt_embs  = EMB_MODEL.encode(gt_claims, convert_to_tensor=True, show_progress_bar=False)
    ctx_embs = EMB_MODEL.encode(ctx_sents, convert_to_tensor=True, show_progress_bar=False)

    covered, missing = 0, []
    for i, claim in enumerate(gt_claims):
        max_sim = float(util.cos_sim(gt_embs[i], ctx_embs).max())
        if max_sim >= threshold:
            covered += 1
        else:
            missing.append(claim)

    return {'score': covered / len(gt_claims),
            'covered': covered, 'total': len(gt_claims), 'missing': missing}


print('=== Context Recall: Good vs Noisy vs Poor Retriever ===\n')
print('Context recall measures: did the retriever find everything needed to answer?\n')

cr_scores = {name: [] for name in STRATEGIES}

for g in GOLDEN:
    for strat_name, retr in STRATEGIES.items():
        retrieved_ids  = get_retrieved_ids(retr, g['id'], k=5)
        context_text   = ' '.join(DOCS[did] for did in retrieved_ids if did in DOCS)
        result         = context_recall_mock(g['ground_truth'], context_text)
        cr_scores[strat_name].append(result['score'])

print(f'{"Query":<50} ' + '  '.join(f'{n[:7]:<10}' for n in STRATEGIES))
print('-' * 85)
for i, g in enumerate(GOLDEN):
    row = '  '.join(f'{cr_scores[n][i]:<10.2f}' for n in STRATEGIES)
    print(f'{g["query"][:48]:<50} {row}')

print('-' * 85)
for n in STRATEGIES:
    print(f'  Mean Context Recall ({n:<18}): {np.mean(cr_scores[n]):.3f}')

print()
print('Context recall is metric #2 to instrument.')
print('Low context recall = retriever is failing. No clever prompting can fix it.')
print('The poor retriever scores low because it never finds the relevant doc.')


In [ ]:
# ── Context Precision ─────────────────────────────────────────────────────────
# Of the chunks we retrieved, were the relevant ones ranked at the TOP?
# LLM-judged version of precision@k.
# Uses average precision: a metric that rewards finding relevant docs early.

def context_precision_mock(query: str, retrieved_ids: list,
                           relevant_ids: list) -> dict:
    """
    Average Precision: at each rank, compute precision if the doc at that rank is relevant.
    This rewards finding relevant docs earlier (higher precision at low ranks).
    """
    relevant_set = set(relevant_ids)
    num_relevant = 0
    sum_precision = 0.0

    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_set:
            num_relevant  += 1
            sum_precision += num_relevant / rank  # precision at this rank

    score = sum_precision / len(relevant_ids) if relevant_ids else 0.0
    return {'score': score, 'num_relevant_retrieved': num_relevant,
            'num_relevant_total': len(relevant_ids)}


print('=== Context Precision (Average Precision) ===\n')
print('Context precision rewards finding relevant docs EARLY in the ranking.\n')

cp_scores = {name: [] for name in STRATEGIES}

for g in GOLDEN:
    for strat_name, retr in STRATEGIES.items():
        retrieved_ids = get_retrieved_ids(retr, g['id'], k=5)
        result        = context_precision_mock(g['query'], retrieved_ids, g['relevant_docs'])
        cp_scores[strat_name].append(result['score'])

print(f'{"Query":<50} ' + '  '.join(f'{n[:7]:<10}' for n in STRATEGIES))
print('-' * 85)
for i, g in enumerate(GOLDEN):
    row = '  '.join(f'{cp_scores[n][i]:<10.2f}' for n in STRATEGIES)
    print(f'{g["query"][:48]:<50} {row}')

print('-' * 85)
for n in STRATEGIES:
    print(f'  Mean Context Precision ({n:<18}): {np.mean(cp_scores[n]):.3f}')

print()
print('The noisy retriever finds relevant docs but buries them — low context precision.')
print('Use this metric when irrelevant chunks distract the LLM or inflate token cost.')


In [ ]:
# ── Answer Relevance ──────────────────────────────────────────────────────────
# Did we answer the QUESTION THAT WAS ASKED, or did we drift off-topic?
#
# Clever trick (used by Ragas): generate N synthetic questions FROM the answer,
# then measure how similar they are to the original question.
# If the answer is on-topic, reverse-engineering questions should reproduce the original.

SYNTHETIC_QUESTIONS = {
    # For each answer set, pre-computed reverse questions
    'Faithful': {
        'q1': ['What is the return window for purchases?',
                'Are digital goods refundable?',
                'How do I start a return?'],
        'q2': ['How long does standard shipping take?',
                'What is the cost of expedited shipping?'],
        'q3': ['What uptime guarantee do enterprise customers get?'],
        'q4': ['How many API calls does the free tier allow per day?'],
        'q5': ['Does the company share user data with third parties?'],
        'q6': ['When does subscription access end after cancellation?'],
        'q7': ['How much storage comes with the free tier?'],
    },
    'Hallucinated': {
        'q1': ['What is the refund window?',
                'Is there a restocking fee?',
                'How do I contact support?'],
        'q2': ['Is shipping free?', 'How fast does shipping arrive?'],
        'q3': ['What SLA does enterprise get?'],
        'q4': ['What is the free tier API limit?'],
        'q5': ['What does the company do with user data?'],
        'q6': ['What happens when I cancel?'],
        'q7': ['What storage is included in the free tier?'],
    },
    'Off-topic': {
        'q1': ['Does the company have good prices?',
                'What does the company prioritise?'],
        'q2': ['Does the company use multiple shipping partners?'],
        'q3': ['What kind of plans does enterprise offer?'],
        'q4': ['Is the API developer-friendly?'],
        'q5': ['Does the company care about privacy?'],
        'q6': ['What billing options are available?'],
        'q7': ['Is the free tier a good way to try the product?'],
    },
}


def answer_relevance_mock(original_question: str, synthetic_questions: list) -> float:
    """
    Embed original question and each synthetic question.
    Return average cosine similarity.
    High similarity = answer is on-topic.
    """
    if not synthetic_questions:
        return 0.0
    orig_emb = EMB_MODEL.encode(original_question,
                                convert_to_tensor=True, show_progress_bar=False)
    synth_embs = EMB_MODEL.encode(synthetic_questions,
                                  convert_to_tensor=True, show_progress_bar=False)
    sims = util.cos_sim(orig_emb, synth_embs)[0].cpu().numpy()
    return float(sims.mean())


print('=== Answer Relevance ===\n')
print('Measures: did we answer the question asked, or did we drift off-topic?\n')

ar_scores = defaultdict(list)
print(f'{"Answer type":<18} ' + '  '.join(f'{g["id"]:<8}' for g in GOLDEN) + '  Mean')
print('-' * 80)
for ans_name in ANSWER_SETS:
    scores = []
    for g in GOLDEN:
        synth_qs = SYNTHETIC_QUESTIONS[ans_name][g['id']]
        score    = answer_relevance_mock(g['query'], synth_qs)
        scores.append(score)
        ar_scores[ans_name].append(score)
    row = '  '.join(f'{s:<8.2f}' for s in scores)
    print(f'{ans_name:<18} {row}  {np.mean(scores):.2f}')

print()
print('Faithful and Hallucinated answers both score high — they address the right question.')
print('Off-topic answers score low — they talk about the company in general.')
print('This shows answer relevance complements faithfulness: relevance catches topic drift,')
print('faithfulness catches hallucination. You need both.')


In [ ]:
# ── Answer Correctness ────────────────────────────────────────────────────────
# How accurate is the answer compared to the ground-truth answer?
# Two components:
#   1. Semantic similarity (embedding cosine sim) — are the meanings close?
#   2. Factual accuracy (claims overlap) — do the specific facts match?
# Combined: weighted average

def answer_correctness_mock(generated_answer: str, ground_truth: str,
                            semantic_weight: float = 0.5) -> dict:
    # Semantic similarity
    gen_emb = EMB_MODEL.encode(generated_answer,
                               convert_to_tensor=True, show_progress_bar=False)
    gt_emb  = EMB_MODEL.encode(ground_truth,
                               convert_to_tensor=True, show_progress_bar=False)
    semantic_sim = float(util.cos_sim(gen_emb, gt_emb))

    # Factual accuracy: claim-level overlap
    gen_claims = set(re.split(r'[.,]\s+', generated_answer.lower()))
    gt_claims  = set(re.split(r'[.,]\s+', ground_truth.lower()))
    gen_embs   = EMB_MODEL.encode(list(gen_claims),
                                  convert_to_tensor=True, show_progress_bar=False)
    gt_embs    = EMB_MODEL.encode(list(gt_claims),
                                  convert_to_tensor=True, show_progress_bar=False)
    sims       = util.cos_sim(gen_embs, gt_embs).cpu().numpy()
    # For each GT claim, is there a matching generated claim?
    factual_score = float((sims.max(axis=0) >= 0.75).mean()) if sims.size > 0 else 0.0

    combined = semantic_weight * semantic_sim + (1 - semantic_weight) * factual_score
    return {'score': combined, 'semantic_similarity': semantic_sim,
            'factual_accuracy': factual_score}


print('=== Answer Correctness ===\n')
print('Compares generated answer against ground-truth answer.')
print('Requires labeled ground-truth answers (the most expensive metric).\n')

ac_scores = defaultdict(list)
print(f'{"Answer type":<18} {"Semantic sim":<16} {"Factual acc":<14} {"Combined"}')
print('-' * 65)
for ans_name, answers in ANSWER_SETS.items():
    sem_sims, fact_accs, combineds = [], [], []
    for g in GOLDEN:
        result = answer_correctness_mock(answers[g['id']], g['ground_truth'])
        sem_sims.append(result['semantic_similarity'])
        fact_accs.append(result['factual_accuracy'])
        combineds.append(result['score'])
        ac_scores[ans_name].append(result['score'])
    print(f'{ans_name:<18} {np.mean(sem_sims):<16.3f} {np.mean(fact_accs):<14.3f} {np.mean(combineds):.3f}')

print()
print('Faithful answers score highest on correctness.')
print('Hallucinated answers may score moderate semantic similarity (they sound similar)')
print('but low factual accuracy (the specific numbers are wrong).')


---
## 14. The Two-Metric Diagnostic

You don't need all five Floor 2 metrics on day one.
Start with just **two** — they localise which half of the pipeline failed:

```
               Faithfulness HIGH        Faithfulness LOW
                                ↓              ↓
Context          Answer is GOOD      Answer is HALLUCINATED
Recall HIGH      (both pipeline       (LLM made stuff up
                  halves work)         despite good context)

Context          Retriever FAILED     BOTH failed
Recall LOW       (LLM was honest      (retriever missed the
                  about "I don't       doc AND LLM hallucinated)
                  know")
```

This two-metric combo is the diagnostic equivalent of a multimeter:
you can tell which half of your RAG pipeline to fix first without reading any output manually.


In [ ]:
# ── Two-metric diagnostic scatter plot ───────────────────────────────────────
# For each answer type + retrieval strategy combo, compute:
#   x = context recall (retriever quality)
#   y = faithfulness   (generator quality)
# Plot to show which quadrant each combination lands in.

COMBOS = [
    ('Good retriever + Faithful',    RETRIEVALS_GOOD, FAITHFUL_ANSWERS,     '#1565C0'),
    ('Noisy retriever + Faithful',   RETRIEVALS_NOISY,FAITHFUL_ANSWERS,     '#1E88E5'),
    ('Poor retriever + Faithful',    RETRIEVALS_POOR, FAITHFUL_ANSWERS,     '#90CAF9'),
    ('Good retriever + Hallucinated',RETRIEVALS_GOOD, HALLUCINATED_ANSWERS, '#E53935'),
    ('Good retriever + Off-topic',   RETRIEVALS_GOOD, OFF_TOPIC_ANSWERS,    '#F57F17'),
]

points = []
for label, retr, answers, color in COMBOS:
    ctx_recalls, faith_vals = [], []
    for g in GOLDEN:
        retrieved_ids = get_retrieved_ids(retr, g['id'], k=5)
        context_text  = ' '.join(DOCS[did] for did in retrieved_ids if did in DOCS)
        cr = context_recall_mock(g['ground_truth'], context_text)['score']
        f  = faithfulness_mock(answers[g['id']], context_text)['score']
        ctx_recalls.append(cr)
        faith_vals.append(f)
    points.append({'label': label, 'cr': np.mean(ctx_recalls),
                   'faith': np.mean(faith_vals), 'color': color})

fig, ax = plt.subplots(figsize=(10, 7))
for p in points:
    ax.scatter(p['cr'], p['faith'], s=200, color=p['color'], zorder=5, alpha=0.9)
    ax.annotate(p['label'], (p['cr'], p['faith']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

# Quadrant lines
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.4)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4)

# Quadrant labels
ax.text(0.15, 0.85, 'Retriever failed\n(LLM was honest)', ha='center',
        fontsize=10, color='gray', transform=ax.transAxes)
ax.text(0.75, 0.85, 'All good!', ha='center',
        fontsize=10, color='#1565C0', fontweight='bold', transform=ax.transAxes)
ax.text(0.15, 0.15, 'Both failed', ha='center',
        fontsize=10, color='#E53935', fontweight='bold', transform=ax.transAxes)
ax.text(0.75, 0.15, 'LLM hallucinated\n(despite good context)', ha='center',
        fontsize=10, color='#E53935', transform=ax.transAxes)

ax.set_xlabel('Context Recall (retriever quality)', fontsize=12)
ax.set_ylabel('Faithfulness (generator quality)', fontsize=12)
ax.set_title('Two-Metric Diagnostic\n'
             'Localise failure: which half of the RAG pipeline is broken?',
             fontweight='bold', fontsize=12)
ax.set_xlim(-0.05, 1.15)
ax.set_ylim(-0.05, 1.15)
plt.tight_layout()
plt.show()

print('Read this chart like a multimeter:')
print('  Top-right  : retriever AND generator working — ship it')
print('  Top-left   : retriever failing — fix your embedding model or chunking strategy')
print('  Bottom-right: LLM hallucinating despite good context — fix your synthesis prompt')
print('  Bottom-left : both failing — start with the retriever (it unlocks everything else)')


In [ ]:
# ── Full metric cheat sheet visualisation ────────────────────────────────────
METRIC_INFO = [
    # (name, floor, priority, requires_labels, llm_needed, when_to_use)
    ('Recall@K',          1, 2, True,  False, 'LLM needs 1 good chunk'),
    ('Precision@K',       1, 3, True,  False, 'Small context window'),
    ('MRR',               1, 3, True,  False, 'Ranking matters to LLM'),
    ('NDCG',              1, 4, True,  False, 'Multi-level relevance labels'),
    ('Faithfulness',      2, 1, False, True,  'Always — #1 metric'),
    ('Context Recall',    2, 1, True,  True,  'Always — #2 metric'),
    ('Context Precision', 2, 2, True,  True,  'Noisy retrieval / long prompts'),
    ('Answer Relevance',  2, 2, False, True,  'Drift / off-topic answers'),
    ('Answer Correctness',2, 3, True,  True,  'Factual accuracy required'),
    ('Human Preference',  3, 1, False, False, 'Gold standard; slow and costly'),
    ('Task Completion',   3, 1, False, False, 'Production; hardest to define'),
    ('Citation Accuracy', 3, 2, False, False, 'RAG with cited sources'),
]

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)
ax.axis('off')

FLOOR_COLORS = {1: '#E3F2FD', 2: '#FFF8E1', 3: '#F3E5F5'}
FLOOR_LABELS = {1: 'Floor 1: Retrieval Metrics\n(pure math, fast, cheap)',
                2: 'Floor 2: RAG-Specific\n(LLM judge)',
                3: 'Floor 3: End-to-End\n(business scoreboard)'}

col_positions = {1: (0.2, 3.0, 3.5), 2: (4.0, 3.0, 3.5), 3: (8.3, 3.0, 3.5)}

for floor, (x, y, w) in col_positions.items():
    rect = plt.Rectangle((x - 0.1, 0.1), w + 0.2, 3.7,
                          fc=FLOOR_COLORS[floor], ec='#BDBDBD', lw=1.5, zorder=1)
    ax.add_patch(rect)
    ax.text(x + w/2, 3.8, FLOOR_LABELS[floor], ha='center', fontsize=9,
            fontweight='bold', color='#424242')

floor_metrics = {1: [], 2: [], 3: []}
for m in METRIC_INFO:
    floor_metrics[m[1]].append(m)

for floor, metrics in floor_metrics.items():
    x, _, w = col_positions[floor]
    for j, m in enumerate(metrics):
        y_pos = 3.3 - j * 0.7
        prio_color = '#1565C0' if m[2]==1 else ('#F57F17' if m[2]==2 else '#9E9E9E')
        ax.text(x + 0.1, y_pos, f'P{m[2]}', fontsize=8, color='white',
                bbox=dict(fc=prio_color, ec='none', pad=2), fontweight='bold')
        ax.text(x + 0.55, y_pos, m[0], fontsize=9, va='center', fontweight='bold')
        ax.text(x + 0.55, y_pos - 0.22, m[5], fontsize=7.5, va='center', color='#616161')
        label_parts = []
        if m[3]:
            label_parts.append('labels needed')
        if m[4]:
            label_parts.append('LLM judge')
        if label_parts:
            ax.text(x + w - 0.15, y_pos, ', '.join(label_parts),
                    fontsize=7, va='center', ha='right', color='#9E9E9E', style='italic')

legend_handles = [
    mpatches.Patch(color='#1565C0', label='Priority 1 — instrument first'),
    mpatches.Patch(color='#F57F17', label='Priority 2 — add next'),
    mpatches.Patch(color='#9E9E9E', label='Priority 3 — when mature'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9,
          bbox_to_anchor=(1.0, -0.02))
ax.set_title('RAG Evaluation Metric Map', fontweight='bold', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print('Instrument in this order:')
print('  1. Faithfulness + Context Recall (localise the failure)')
print('  2. Recall@K (cheapest retrieval check; no LLM needed)')
print('  3. Answer Relevance (catch topic drift)')
print('  4. Rest of Floor 1 + Floor 2 (as the system matures)')
print('  5. Floor 3 (when you need business-level accountability)')


---
## Key Takeaways

1. **"Is my RAG good?" is three questions, not one.** Is the retriever finding the right
   docs? Is the generator using them faithfully? Is the end-to-end system helping users?
   Each question needs its own metrics. Mixing them up is why people get confused.

2. **Recall@K is your first retrieval sanity check** — but it lies by omission.
   Recall@5 = 1.0 just means the right doc is somewhere in the bag.
   MRR tells you if it was at rank 1 or rank 5, and those are very different outcomes.

3. **Faithfulness is your #1 metric to instrument.** Break the answer into claims,
   check each claim against the context. Hallucinated claims are the most dangerous
   failure mode — no error is raised, no warning logged.

4. **Context Recall is your #2 metric.** It localises retriever failures precisely.
   Low context recall = your retriever is missing information. No clever prompting can recover it.
   Fix the retriever first.

5. **The two-metric diagnostic is your multimeter.** Plot Faithfulness (y) vs. Context
   Recall (x). Top-right = working. Top-left = fix the retriever. Bottom-right = fix the
   synthesis prompt. Bottom-left = fix the retriever first (it unlocks everything else).

6. **LLM-as-judge enables evaluation without labeling thousands of examples.**
   Break answers into atomic claims and judge each one. This scales evaluation to
   hundreds of queries without a human reading every response.

7. **Answer Relevance catches a different failure than Faithfulness.**
   A hallucinated answer and an off-topic answer both fail — but for different reasons.
   Faithfulness catches "the LLM made stuff up." Relevance catches "the LLM answered
   a different question."

8. **Floor 1 metrics are necessary but not sufficient.** Perfect Recall@5 can coexist
   with terrible answers. You need Floor 2 metrics to know whether the LLM actually
   used what the retriever found.

---

*Up next — Lesson 11.2: LLM-as-judge — how to trust the judge, build a test set without*
*labeling 1,000 examples, and choose between Ragas, TruLens, and DeepEval.*
